# Step 2: Data Collection & Understanding
**Capstone Project — Cybersecurity: Detect Anomalies in Network Traffic**

Dataset: CICIDS2017 (CIC Intrusion Detection Evaluation Dataset 2017)  
Source: Canadian Institute for Cybersecurity, University of New Brunswick  
Kaggle: https://www.kaggle.com/datasets/cicdataset/cicids2017

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Load Dataset

CICIDS2017 is distributed as 8 separate CSV files (one per day/session). We download via Kaggle API and concatenate into a single DataFrame.

**Setup:** Upload your `kaggle.json` API token to Colab before running this cell.

In [ ]:
# --- Kaggle API Setup ---
# Step 1: Go to https://www.kaggle.com/settings → API section → click "Create New Token"
#         This downloads kaggle.json to your computer.
# Step 2: Run this cell — a file picker will open. Select that kaggle.json file.
# Step 3: You should see "Authenticated as: your_username" before proceeding.

import os, json as _json
from google.colab import files

uploaded = files.upload()  # file picker opens here

if 'kaggle.json' not in uploaded:
    raise FileNotFoundError(
        'No kaggle.json received. Make sure you selected the right file.'
    )

creds = _json.loads(uploaded['kaggle.json'])
assert 'username' in creds and 'key' in creds, (
    'kaggle.json is missing username or key — '
    're-download a fresh token from https://www.kaggle.com/settings under API'
)

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_path = os.path.join(kaggle_dir, 'kaggle.json')

with open(kaggle_path, 'wb') as f:
    f.write(uploaded['kaggle.json'])

os.chmod(kaggle_path, 0o600)
print(f'Authenticated as: {creds["username"]}')
print('Ready to download dataset.')


In [ ]:
# --- Download CICIDS2017 ---
# The original cicdataset/cicids2017 has been removed from Kaggle.
# Try the mirrors below in order — run whichever works first, then skip to cell 6.

import os, subprocess

os.makedirs('./cicids2017', exist_ok=True)

def kaggle_download(dataset_id):
    print(f'Trying: {dataset_id} ...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', dataset_id,
         '--unzip', '-p', './cicids2017'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'Download complete: {dataset_id}')
        return True
    print(f'Failed ({result.stderr.strip()[:120]})')
    return False

# Try mirrors in order of reliability
mirrors = [
    'chethuhn/network-intrusion-dataset',     # most cited alternative
    'naveengill/cicids2017-dataset',           # updated Dec 2024
    'sweety18/cicids2017-full-dataset',        # labelled "full dataset"
    'hcavsi/cicids2017-dataset',              # additional mirror
]

success = False
for mirror in mirrors:
    if kaggle_download(mirror):
        success = True
        break

if not success:
    print()
    print('All Kaggle mirrors failed. Use manual upload instead:')
    print()
    print('1. Download from Kaggle in your browser: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset')
    print('   OR from UNB directly: https://www.unb.ca/cic/datasets/ids-2017.html')
    print()
    print('2. Upload to Google Drive, then run:')
    print('   from google.colab import drive')
    print('   drive.mount("/content/drive")')
    print('   # Then update the glob path in cell 6 to point to your Drive folder')


In [ ]:
# List all CSV files in the downloaded folder
csv_files = sorted(glob.glob('./cicids2017/**/*.csv', recursive=True))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    size_mb = os.path.getsize(f) / (1024 ** 2)
    print(f'  {os.path.basename(f):<60} {size_mb:.1f} MB')

In [ ]:
# Load and concatenate all CSVs with memory optimisation
# CICIDS2017 at full float64 uses ~1.7GB RAM — float32 halves that.
# A stratified 500k-row sample is used if RAM is still tight;
# this is academically acceptable and noted in the report.

SAMPLE_SIZE = 500_000   # set to None to load everything (may crash on free Colab tier)
RANDOM_SEED = 42

def optimise_dtypes(df):
    """Cast float64 -> float32 and downcast integers to save ~50% RAM."""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

dfs = []
for f in csv_files:
    df_part = pd.read_csv(f, encoding='utf-8', low_memory=False)
    df_part.columns = df_part.columns.str.strip()
    df_part['source_file'] = os.path.basename(f)
    df_part = optimise_dtypes(df_part)
    dfs.append(df_part)
    mem_mb = df_part.memory_usage(deep=True).sum() / 1024**2
    print(f'  Loaded {os.path.basename(f)}: {df_part.shape}  ({mem_mb:.0f} MB)')

df = pd.concat(dfs, ignore_index=True)
del dfs

print()
print(f'Full combined dataset : {df.shape[0]:,} rows x {df.shape[1]} columns')
total_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memory usage          : {total_mb:.0f} MB')

# Stratified sample if dataset is large (keeps class proportions intact)
if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    label_col_temp = 'Label' if 'Label' in df.columns else df.columns[-2]
    df[label_col_temp] = df[label_col_temp].astype(str).str.strip()
    df = (
        df.groupby(label_col_temp, group_keys=False)
          .apply(lambda x: x.sample(
              min(len(x), max(1, int(SAMPLE_SIZE * len(x) / len(df)))),
              random_state=RANDOM_SEED
          ))
          .reset_index(drop=True)
    )
    sample_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f'Sampled to            : {len(df):,} rows  ({sample_mb:.0f} MB)')
    print('Note: stratified sample preserves class proportions — valid for EDA and modelling.')

print()
print(f'Final shape: {df.shape}')


## 3. Initial Inspection

In [ ]:
df.head(5)

In [ ]:
df.info(verbose=True, show_counts=True)

## 4. Duplicate Column Check

CICIDS2017 is known to contain a duplicate `Fwd Header Length` column.

In [ ]:
# Identify duplicate column names
dup_cols = df.columns[df.columns.duplicated()].tolist()
print(f'Duplicate column names: {dup_cols}')

# Drop the second occurrence of any duplicate
df = df.loc[:, ~df.columns.duplicated()]
print(f'Shape after removing duplicates: {df.shape}')

## 5. Feature Type Summary

In [ ]:
# Separate feature columns from metadata/label
meta_cols    = ['source_file']
label_col    = 'Label'
feature_cols = [c for c in df.columns if c not in meta_cols + [label_col]]

# Categorise by dtype
numeric_cols  = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
object_cols   = df[feature_cols].select_dtypes(include=['object']).columns.tolist()

print(f'Total feature columns : {len(feature_cols)}')
print(f'Numeric features      : {len(numeric_cols)}')
print(f'Object/string features: {len(object_cols)}')
if object_cols:
    print(f'  Object columns: {object_cols}')

# Sub-categorise numerics by role
flag_features   = [c for c in numeric_cols if 'Flag' in c]
iat_features    = [c for c in numeric_cols if 'IAT' in c]
length_features = [c for c in numeric_cols if 'Length' in c or 'Bytes' in c or 'bytes' in c]
packet_features = [c for c in numeric_cols if 'Packet' in c or 'packet' in c]
rate_features   = [c for c in numeric_cols if '/s' in c or 'Rate' in c or 'rate' in c]
other_numeric   = [c for c in numeric_cols
                   if c not in flag_features + iat_features + length_features
                   + packet_features + rate_features]

type_summary = pd.DataFrame({
    'Feature Group': ['Flag Count', 'Inter-Arrival Time (IAT)',
                      'Byte/Length', 'Packet Count', 'Rate', 'Other Numeric'],
    'Count': [len(flag_features), len(iat_features), len(length_features),
              len(packet_features), len(rate_features), len(other_numeric)],
    'Examples': [
        ', '.join(flag_features[:3]),
        ', '.join(iat_features[:3]),
        ', '.join(length_features[:3]),
        ', '.join(packet_features[:3]),
        ', '.join(rate_features[:3]),
        ', '.join(other_numeric[:3])
    ]
})
display(type_summary)

## 6. Class Distribution

In [ ]:
# Normalise label strings — strip whitespace, consistent capitalisation
df[label_col] = df[label_col].str.strip()

label_counts = df[label_col].value_counts()
label_pct    = df[label_col].value_counts(normalize=True).mul(100).round(3)

label_summary = pd.DataFrame({'Count': label_counts, 'Percentage (%)': label_pct})
print('Class distribution:')
display(label_summary)

# Map to attack category groups for grouped view
category_map = {
    'BENIGN'                    : 'Benign',
    'DDoS'                      : 'DDoS',
    'DoS Hulk'                  : 'DoS',
    'DoS GoldenEye'             : 'DoS',
    'DoS slowloris'             : 'DoS',
    'DoS Slowhttptest'          : 'DoS',
    'Heartbleed'                : 'DoS',
    'FTP-Patator'               : 'Brute Force',
    'SSH-Patator'               : 'Brute Force',
    'Web Attack \xe2\x80\x93 Brute Force'  : 'Web Attack',
    'Web Attack – Brute Force'  : 'Web Attack',
    'Web Attack \xe2\x80\x93 XSS'          : 'Web Attack',
    'Web Attack – XSS'          : 'Web Attack',
    'Web Attack \xe2\x80\x93 Sql Injection' : 'Web Attack',
    'Web Attack – Sql Injection': 'Web Attack',
    'Infiltration'              : 'Infiltration',
    'Bot'                       : 'Botnet',
    'PortScan'                  : 'Probe',
}
df['attack_category'] = df[label_col].map(category_map).fillna('Other')

cat_counts = df['attack_category'].value_counts()
print('\nGrouped attack category distribution:')
display(pd.DataFrame({'Count': cat_counts,
                      'Percentage (%)': cat_counts / len(df) * 100}).round(3))

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full label distribution (log scale due to imbalance)
label_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('muted', len(label_counts)))
axes[0].set_xscale('log')
axes[0].set_title('Record Count by Label (log scale)')
axes[0].set_xlabel('Count (log scale)')

# Grouped category pie
axes[1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('muted', len(cat_counts)), startangle=140)
axes[1].set_title('Grouped Attack Category Distribution')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Missing Value & Infinite Value Analysis

In [ ]:
# NaN check
missing = df[feature_cols].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'Columns with NaN values: {len(missing)}')
if len(missing) > 0:
    display(pd.DataFrame({
        'NaN Count': missing,
        'NaN %': (missing / len(df) * 100).round(4)
    }))

# Infinite value check (common in CICIDS2017 due to division-by-zero in flow rate features)
inf_counts = {}
for col in numeric_cols:
    n_inf = np.isinf(df[col].replace([np.inf, -np.inf], np.nan)).sum()
    # re-check properly
    n_inf = ((df[col] == np.inf) | (df[col] == -np.inf)).sum()
    if n_inf > 0:
        inf_counts[col] = n_inf

print(f'\nColumns with Inf values: {len(inf_counts)}')
if inf_counts:
    inf_df = pd.DataFrame.from_dict(inf_counts, orient='index', columns=['Inf Count'])
    inf_df['Inf %'] = (inf_df['Inf Count'] / len(df) * 100).round(4)
    display(inf_df.sort_values('Inf Count', ascending=False))

print(f'\nTotal problematic values: {missing.sum() + sum(inf_counts.values())}')

## 8. Descriptive Statistics

In [ ]:
# Replace inf with NaN for stats calculation
df_stats = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

desc = df_stats.describe().T
desc['skewness'] = df_stats.skew()
desc['kurtosis'] = df_stats.kurtosis()
desc['zero_pct'] = ((df_stats == 0).sum() / len(df) * 100).round(2)

print('Descriptive statistics for all numeric features:')
display(desc.round(4))

## 9. Zero-Variance Feature Check

In [ ]:
# Features with zero or near-zero variance add no signal and should be dropped in Step 3
zero_var_cols = [c for c in numeric_cols if df[c].nunique() <= 1]
near_zero_var = [c for c in numeric_cols
                 if df[c].nunique() > 1 and df[c].value_counts(normalize=True).iloc[0] > 0.99]

print(f'Zero-variance columns (constant): {zero_var_cols}')
print(f'Near-zero variance columns (>99% one value): {near_zero_var}')

## 10. Outlier Detection (IQR Method)

In [ ]:
def count_outliers_iqr(series):
    s = series.replace([np.inf, -np.inf], np.nan).dropna()
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        return 0
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((s < lower) | (s > upper)).sum()

# Sample for speed if dataset is large
sample_df = df[numeric_cols].sample(min(100_000, len(df)), random_state=42)

outlier_summary = pd.DataFrame({
    'Outlier Count (IQR)': [count_outliers_iqr(sample_df[c]) for c in numeric_cols],
    'Outlier %': [(count_outliers_iqr(sample_df[c]) / len(sample_df) * 100) for c in numeric_cols],
    'Median': [sample_df[c].median() for c in numeric_cols],
    'Max': [sample_df[c].replace([np.inf, -np.inf], np.nan).max() for c in numeric_cols],
}, index=numeric_cols).sort_values('Outlier %', ascending=False)

print('Top 15 features by outlier prevalence (IQR, 100k sample):')
display(outlier_summary.head(15).round(2))

In [ ]:
# Box plots for top outlier-prone features
top_outlier_features = outlier_summary.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(top_outlier_features):
    data = sample_df[col].replace([np.inf, -np.inf], np.nan).dropna()
    axes[i].boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(col, fontsize=8)
    axes[i].set_xticks([])

plt.suptitle('Box Plots — Top Features by Outlier Prevalence (100k sample)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Feature Distributions: Benign vs Attack

In [ ]:
# Select representative features across groups for distribution comparison
key_features = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Flow Packets/s', 'Fwd IAT Mean',
    'PSH Flag Count', 'SYN Flag Count'
]
# Filter to only features that exist in this dataset
key_features = [f for f in key_features if f in df.columns]

benign_mask = df['attack_category'] == 'Benign'

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(key_features[:8]):
    ax = axes[i]
    benign_vals = df.loc[benign_mask, col].replace([np.inf, -np.inf], np.nan).dropna()
    attack_vals = df.loc[~benign_mask, col].replace([np.inf, -np.inf], np.nan).dropna()
    # Cap at 99th percentile for readability
    cap = np.percentile(pd.concat([benign_vals, attack_vals]), 99)
    benign_vals.clip(upper=cap).hist(bins=50, alpha=0.6, label='Benign',
                                      color='steelblue', ax=ax, density=True)
    attack_vals.clip(upper=cap).hist(bins=50, alpha=0.6, label='Attack',
                                      color='firebrick', ax=ax, density=True)
    ax.set_title(col, fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions: Benign vs Attack (capped at 99th percentile)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Correlation with Label

In [ ]:
# Binary label for correlation
df['label_binary'] = (df['attack_category'] != 'Benign').astype(int)

df_corr = df[numeric_cols + ['label_binary']].replace([np.inf, -np.inf], np.nan)

label_corr = df_corr.corr()['label_binary'].drop('label_binary').abs().sort_values(ascending=False)
top_corr_features = label_corr.head(20).index.tolist()

print('Top 15 features by absolute correlation with binary attack label:')
display(label_corr.head(15).round(4))

plt.figure(figsize=(14, 10))
sns.heatmap(
    df_corr[top_corr_features + ['label_binary']].corr(),
    annot=False, cmap='coolwarm', center=0, linewidths=0.3
)
plt.title('Correlation Heatmap — Top 20 Features vs Binary Label')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Data Dictionary

In [ ]:
data_dict = [
    # Flow identification
    {'Feature': 'Destination Port',      'Type': 'Integer', 'Group': 'Flow ID',          'Range': '0–65535',        'Description': 'Destination port number of the network flow'},
    {'Feature': 'Protocol',              'Type': 'Integer', 'Group': 'Flow ID',          'Range': '0–255',          'Description': 'IP protocol number (e.g., 6=TCP, 17=UDP, 0=HOPOPT)'},
    # Flow timing
    {'Feature': 'Flow Duration',         'Type': 'Float',   'Group': 'Flow Timing',      'Range': '>= 0 (µs)',      'Description': 'Duration of the flow in microseconds'},
    {'Feature': 'Flow IAT Mean',         'Type': 'Float',   'Group': 'Flow Timing',      'Range': '>= 0 (µs)',      'Description': 'Mean inter-arrival time between flow packets'},
    {'Feature': 'Flow IAT Std',          'Type': 'Float',   'Group': 'Flow Timing',      'Range': '>= 0 (µs)',      'Description': 'Standard deviation of inter-arrival times'},
    {'Feature': 'Flow IAT Max',          'Type': 'Float',   'Group': 'Flow Timing',      'Range': '>= 0 (µs)',      'Description': 'Maximum inter-arrival time in the flow'},
    {'Feature': 'Flow IAT Min',          'Type': 'Float',   'Group': 'Flow Timing',      'Range': '>= 0 (µs)',      'Description': 'Minimum inter-arrival time in the flow'},
    # Forward packet stats
    {'Feature': 'Total Fwd Packets',     'Type': 'Integer', 'Group': 'Fwd Packets',      'Range': '>= 0',           'Description': 'Total packets in the forward direction'},
    {'Feature': 'Fwd Packet Length Mean','Type': 'Float',   'Group': 'Fwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Mean size of packets in the forward direction'},
    {'Feature': 'Fwd Packet Length Max', 'Type': 'Float',   'Group': 'Fwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Maximum packet size in forward direction'},
    {'Feature': 'Fwd Packet Length Min', 'Type': 'Float',   'Group': 'Fwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Minimum packet size in forward direction'},
    {'Feature': 'Fwd Packet Length Std', 'Type': 'Float',   'Group': 'Fwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Std deviation of forward packet sizes'},
    {'Feature': 'Fwd IAT Total',         'Type': 'Float',   'Group': 'Fwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Total time between forward packets'},
    {'Feature': 'Fwd IAT Mean',          'Type': 'Float',   'Group': 'Fwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Mean time between forward packets'},
    {'Feature': 'Fwd IAT Std',           'Type': 'Float',   'Group': 'Fwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Std dev of time between forward packets'},
    {'Feature': 'Fwd IAT Max',           'Type': 'Float',   'Group': 'Fwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Max time between forward packets'},
    {'Feature': 'Fwd IAT Min',           'Type': 'Float',   'Group': 'Fwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Min time between forward packets'},
    # Backward packet stats
    {'Feature': 'Total Backward Packets','Type': 'Integer', 'Group': 'Bwd Packets',      'Range': '>= 0',           'Description': 'Total packets in the backward direction'},
    {'Feature': 'Bwd Packet Length Mean','Type': 'Float',   'Group': 'Bwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Mean size of backward packets'},
    {'Feature': 'Bwd Packet Length Max', 'Type': 'Float',   'Group': 'Bwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Maximum backward packet size'},
    {'Feature': 'Bwd Packet Length Min', 'Type': 'Float',   'Group': 'Bwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Minimum backward packet size'},
    {'Feature': 'Bwd Packet Length Std', 'Type': 'Float',   'Group': 'Bwd Packets',      'Range': '>= 0 (bytes)',   'Description': 'Std dev of backward packet sizes'},
    {'Feature': 'Bwd IAT Total',         'Type': 'Float',   'Group': 'Bwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Total time between backward packets'},
    {'Feature': 'Bwd IAT Mean',          'Type': 'Float',   'Group': 'Bwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Mean time between backward packets'},
    {'Feature': 'Bwd IAT Std',           'Type': 'Float',   'Group': 'Bwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Std dev of time between backward packets'},
    {'Feature': 'Bwd IAT Max',           'Type': 'Float',   'Group': 'Bwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Max time between backward packets'},
    {'Feature': 'Bwd IAT Min',           'Type': 'Float',   'Group': 'Bwd Timing',       'Range': '>= 0 (µs)',      'Description': 'Min time between backward packets'},
    # Byte volume
    {'Feature': 'Total Length of Fwd Packets','Type':'Integer','Group':'Byte Volume',    'Range': '>= 0 (bytes)',   'Description': 'Total bytes in forward packets'},
    {'Feature': 'Total Length of Bwd Packets','Type':'Integer','Group':'Byte Volume',    'Range': '>= 0 (bytes)',   'Description': 'Total bytes in backward packets'},
    {'Feature': 'Flow Bytes/s',          'Type': 'Float',   'Group': 'Rate',             'Range': '>= 0; can be Inf','Description': 'Bytes per second of the flow; Inf when duration=0'},
    {'Feature': 'Flow Packets/s',        'Type': 'Float',   'Group': 'Rate',             'Range': '>= 0; can be Inf','Description': 'Packets per second; Inf when duration=0'},
    {'Feature': 'Fwd Packets/s',         'Type': 'Float',   'Group': 'Rate',             'Range': '>= 0',           'Description': 'Forward packets per second'},
    {'Feature': 'Bwd Packets/s',         'Type': 'Float',   'Group': 'Rate',             'Range': '>= 0',           'Description': 'Backward packets per second'},
    # Header sizes
    {'Feature': 'Fwd Header Length',     'Type': 'Integer', 'Group': 'Header',           'Range': '>= 0 (bytes)',   'Description': 'Total bytes used for headers in forward direction'},
    {'Feature': 'Bwd Header Length',     'Type': 'Integer', 'Group': 'Header',           'Range': '>= 0 (bytes)',   'Description': 'Total bytes used for headers in backward direction'},
    # TCP flags
    {'Feature': 'FIN Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with FIN flag set'},
    {'Feature': 'SYN Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with SYN flag; high = SYN flood indicator'},
    {'Feature': 'RST Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with RST flag set'},
    {'Feature': 'PSH Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with PSH flag set'},
    {'Feature': 'ACK Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with ACK flag set'},
    {'Feature': 'URG Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with URG flag set'},
    {'Feature': 'CWE Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with CWE flag set'},
    {'Feature': 'ECE Flag Count',        'Type': 'Integer', 'Group': 'TCP Flags',        'Range': '0–N',            'Description': 'Number of packets with ECE flag set'},
    # Subflow & window
    {'Feature': 'Subflow Fwd Packets',   'Type': 'Integer', 'Group': 'Subflow',          'Range': '>= 0',           'Description': 'Average forward packets in a subflow'},
    {'Feature': 'Subflow Fwd Bytes',     'Type': 'Integer', 'Group': 'Subflow',          'Range': '>= 0 (bytes)',   'Description': 'Average forward bytes in a subflow'},
    {'Feature': 'Subflow Bwd Packets',   'Type': 'Integer', 'Group': 'Subflow',          'Range': '>= 0',           'Description': 'Average backward packets in a subflow'},
    {'Feature': 'Subflow Bwd Bytes',     'Type': 'Integer', 'Group': 'Subflow',          'Range': '>= 0 (bytes)',   'Description': 'Average backward bytes in a subflow'},
    {'Feature': 'Init_Win_bytes_forward','Type': 'Integer', 'Group': 'TCP Window',       'Range': '-1 to 65535',    'Description': 'Initial TCP window bytes in forward direction; -1 if not applicable'},
    {'Feature': 'Init_Win_bytes_backward','Type':'Integer', 'Group': 'TCP Window',       'Range': '-1 to 65535',    'Description': 'Initial TCP window bytes in backward direction'},
    {'Feature': 'act_data_pkt_fwd',      'Type': 'Integer', 'Group': 'Active/Idle',      'Range': '>= 0',           'Description': 'Count of packets with at least 1 byte of TCP data payload (forward)'},
    {'Feature': 'min_seg_size_forward',  'Type': 'Integer', 'Group': 'Active/Idle',      'Range': '>= 0 (bytes)',   'Description': 'Minimum segment size in forward direction'},
    {'Feature': 'Active Mean',           'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Mean time the flow was active before becoming idle'},
    {'Feature': 'Active Std',            'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Std dev of active periods'},
    {'Feature': 'Active Max',            'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Maximum active period duration'},
    {'Feature': 'Active Min',            'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Minimum active period duration'},
    {'Feature': 'Idle Mean',             'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Mean time the flow was idle before becoming active'},
    {'Feature': 'Idle Std',              'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Std dev of idle periods'},
    {'Feature': 'Idle Max',              'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Maximum idle period duration'},
    {'Feature': 'Idle Min',              'Type': 'Float',   'Group': 'Active/Idle',      'Range': '>= 0 (µs)',      'Description': 'Minimum idle period duration'},
    # Aggregate/ratio
    {'Feature': 'Down/Up Ratio',         'Type': 'Float',   'Group': 'Ratio',            'Range': '>= 0',           'Description': 'Ratio of download to upload bytes'},
    {'Feature': 'Average Packet Size',   'Type': 'Float',   'Group': 'Ratio',            'Range': '>= 0 (bytes)',   'Description': 'Mean size of all packets in the flow'},
    {'Feature': 'Avg Fwd Segment Size',  'Type': 'Float',   'Group': 'Ratio',            'Range': '>= 0 (bytes)',   'Description': 'Average segment size in the forward direction'},
    {'Feature': 'Avg Bwd Segment Size',  'Type': 'Float',   'Group': 'Ratio',            'Range': '>= 0 (bytes)',   'Description': 'Average segment size in the backward direction'},
    # Push flags directional
    {'Feature': 'Fwd PSH Flags',         'Type': 'Binary',  'Group': 'Directional Flags','Range': '0 or 1',         'Description': 'Number of PSH flags in forward direction'},
    {'Feature': 'Bwd PSH Flags',         'Type': 'Binary',  'Group': 'Directional Flags','Range': '0 or 1',         'Description': 'Number of PSH flags in backward direction'},
    {'Feature': 'Fwd URG Flags',         'Type': 'Binary',  'Group': 'Directional Flags','Range': '0 or 1',         'Description': 'Number of URG flags in forward direction'},
    {'Feature': 'Bwd URG Flags',         'Type': 'Binary',  'Group': 'Directional Flags','Range': '0 or 1',         'Description': 'Number of URG flags in backward direction'},
    # Bulk features
    {'Feature': 'Fwd Avg Bytes/Bulk',    'Type': 'Float',   'Group': 'Bulk',             'Range': '>= 0',           'Description': 'Average bytes per bulk in forward direction'},
    {'Feature': 'Fwd Avg Packets/Bulk',  'Type': 'Float',   'Group': 'Bulk',             'Range': '>= 0',           'Description': 'Average packets per bulk in forward direction'},
    {'Feature': 'Fwd Avg Bulk Rate',     'Type': 'Float',   'Group': 'Bulk',             'Range': '>= 0',           'Description': 'Average bulk rate in forward direction'},
    {'Feature': 'Bwd Avg Bytes/Bulk',    'Type': 'Float',   'Group': 'Bulk',             'Range': '>= 0',           'Description': 'Average bytes per bulk in backward direction'},
    {'Feature': 'Bwd Avg Packets/Bulk',  'Type': 'Float',   'Group': 'Bulk',             'Range': '>= 0',           'Description': 'Average packets per bulk in backward direction'},
    {'Feature': 'Bwd Avg Bulk Rate',     'Type': 'Float',   'Group': 'Bulk',             'Range': '>= 0',           'Description': 'Average bulk rate in backward direction'},
    # Label
    {'Feature': 'Label',                 'Type': 'Nominal', 'Group': 'Label',            'Range': '15 distinct values', 'Description': 'Attack type or BENIGN; primary prediction target'},
]

dd = pd.DataFrame(data_dict)
print(f'Data dictionary: {len(dd)} entries')
display(dd)
dd.to_csv('cicids2017_data_dictionary.csv', index=False)
print('Saved to cicids2017_data_dictionary.csv')

## 14. Summary Statistics

In [ ]:
print('=== CICIDS2017 DATASET SUMMARY ===')
print(f'Total records        : {len(df):,}')
print(f'Total columns        : {df.shape[1]}')
print(f'Feature columns      : {len(feature_cols)} (all numeric after dedup)')
print(f'Label column         : Label (15 classes)')
print(f'Missing values (NaN) : {df[feature_cols].isnull().sum().sum()}')
print(f'Zero-variance cols   : {len(zero_var_cols)}')
print(f'Near-zero-var cols   : {len(near_zero_var)}')
print()
print('Attack category distribution:')
for cat, cnt in df['attack_category'].value_counts().items():
    pct = cnt / len(df) * 100
    print(f'  {cat:<16}: {cnt:>9,}  ({pct:.2f}%)')